### Read experimental result data

In [2]:
# read parquet file
import polars as pl
# df = pl.read_parquet("../../experiments/16_10_result.parquet")
df = pl.read_parquet("/Users/hongseokyeong/Desktop/rustモデル/diffusion_seokyeong/twitch_DE.parquet")
print(df)
df.write_csv("df.csv")
# read metadata of the parquet file
import pyarrow.parquet as pq
# meta = pq.read_metadata("../test/result.parquet")
meta = pq.read_metadata("/Users/hongseokyeong/Desktop/rustモデル/diffusion_seokyeong/twitch_DE.parquet")

# You can check versions depending on libraries 
print(meta.metadata)

shape: (524_288, 5)
┌─────┬──────┬───────┬───────┬───────────────┐
│ id  ┆ ip   ┆ us_id ┆ msg   ┆ mean_num_xact │
│ --- ┆ ---  ┆ ---   ┆ ---   ┆ ---           │
│ u64 ┆ u64  ┆ u64   ┆ str   ┆ f64           │
╞═════╪══════╪═══════╪═══════╪═══════════════╡
│ 0   ┆ 9205 ┆ 0     ┆ 30000 ┆ 26.61         │
│ 0   ┆ 9205 ┆ 0     ┆ 20000 ┆ 20.92         │
│ 0   ┆ 9205 ┆ 0     ┆ 31000 ┆ 27.14         │
│ 0   ┆ 9205 ┆ 0     ┆ 21000 ┆ 29.21         │
│ 0   ┆ 9205 ┆ 0     ┆ 01000 ┆ 26.63         │
│ …   ┆ …    ┆ …     ┆ …     ┆ …             │
│ 511 ┆ 2821 ┆ 31    ┆ 32333 ┆ 23.19         │
│ 511 ┆ 2821 ┆ 31    ┆ 33333 ┆ 22.49         │
│ 511 ┆ 2821 ┆ 31    ┆ 03333 ┆ 47.31         │
│ 511 ┆ 2821 ┆ 31    ┆ 13333 ┆ 27.95         │
│ 511 ┆ 2821 ┆ 31    ┆ 23333 ┆ 29.95         │
└─────┴──────┴───────┴───────┴───────────────┘
{b'version_runner': b'0.1.1', b'version_core': b'0.1.1', b'version': b'0.1.0', b'ARROW:schema': b'/////0wBAAAQAAAAAAAKAAwACgAJAAQACgAAABAAAAAAAQQACAAIAAAABAAIAAAABAAAAAUAAADsAAAAsAA

### Compute solutions and/or values of optimal, RAM, and URS policies

In [3]:
EV_BY_OPT = "ev_by_opt"
EV_BY_RAM = "ev_by_ram"
EV_BY_URS = "ev_by_urs"
OPT_MEAN = "mean_of_opt_values"
RAMEV_OPTEV_RATIO = "ram ev / opt ev"
URSEV_OPTEV_RATIO = "urs ev / opt ev"
RAMEV_OPTM_RATIO = "ram ev / opt mean"
URSEV_OPTM_RATIO = "urs ev / opt mean"

MEAN_NUM_XACT = "mean_num_xact"

lf_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", MEAN_NUM_XACT])
    .select([pl.col("us_id"), pl.col("ip"), pl.col("msg").name.prefix("argmax_"), pl.col(MEAN_NUM_XACT).name.prefix("max_")])
)
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))
    .group_by("ip_right", "us_id_right")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])
    .select([MEAN_NUM_XACT, "max_" + MEAN_NUM_XACT])
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias(EV_BY_OPT), pl.col("max_" + MEAN_NUM_XACT).alias(OPT_MEAN)])
)
lf_rob = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max())
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_RAM))
)
lf_urs = (
    df.lazy()
    .select(MEAN_NUM_XACT)
    .mean()
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_URS))
)
eval_df = (
    pl.concat([lf_opt_mod, lf_rob, lf_urs], how="horizontal")
    .with_columns([
        (pl.col(EV_BY_RAM) / pl.col(EV_BY_OPT)).alias(RAMEV_OPTEV_RATIO),
        (pl.col(EV_BY_URS) / pl.col(EV_BY_OPT)).alias(URSEV_OPTEV_RATIO),
        (pl.col(EV_BY_RAM) / pl.col(OPT_MEAN)).alias(RAMEV_OPTM_RATIO),
        (pl.col(EV_BY_URS) / pl.col(OPT_MEAN)).alias(URSEV_OPTM_RATIO),
    ])
).collect().transpose(include_header=True).rename({"column": "key", "column_0": "value"})

/var/folders/bk/xcfgqxl50_l158g0lbjt3_4r0000gn/T/ipykernel_93552/3358191126.py:41: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  pl.concat([lf_opt_mod, lf_rob, lf_urs], how="horizontal")


In [4]:
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))#普通のdfにlf_optで事後確認した最適なmsgを他にip,us_idに適用されたのがあればそれをあわせて平均化。
    .group_by("ip_right", "us_id_right")#他のip,user_stateにもoptのmsgが通用するかという問い
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])#ipとus_idにピッタリなmsg
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias("ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値"),
             pl.col("max_" + MEAN_NUM_XACT).alias("ipとuser_stateに最適なmsg適用後の期待値")])
)
lf_opt_mod.mean().collect()

"ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値",ipとuser_stateに最適なmsg適用後の期待値
f64,f64
54.055612,81.966211


In [5]:
eval_df

key,value
str,f64
"""ev_by_opt""",54.055612
"""mean_of_opt_values""",81.966211
"""ev_by_ram""",77.641016
"""ev_by_urs""",33.203325
"""ram ev / opt ev""",1.436317
"""urs ev / opt ev""",0.614244
"""ram ev / opt mean""",0.947232
"""urs ev / opt mean""",0.405086


ram式message_sequenceごとの平均結果をソート式に並べる

In [6]:
###ram
lf_rob_details = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .sort(by="mean_num_xact", descending=True)
    # 上位5行を取得
).collect()

In [8]:
lf_rob_details

msg,mean_num_xact
str,f64
"""00003""",77.641016
"""00013""",77.032598
"""00023""",75.174082
"""00033""",72.45252
"""00103""",69.998125
…,…
"""33320""",13.56416
"""33330""",13.363965
"""23300""",13.357871


In [7]:
lf_rob_details.select(pl.col("mean_num_xact").mean())

mean_num_xact
f64
33.203325


In [9]:
lf_rob_details[:10].select(pl.col("mean_num_xact").mean())

mean_num_xact
f64
71.3589


In [10]:
# read parquet file
import polars as pl
# user_analysis= pl.read_parquet("../../experiments/user_analysis.parquet")
user_analysis= pl.scan_parquet("/Users/hongseokyeong/Desktop/rustモデル/diffusion_seokyeong/twitch_DE_user_analysis.parquet")


ram上位メッセージ系列だけ抽出→それのみlf_rob_detailsの結果をram適用としてram_msg_us_anに保存

In [11]:
ram_msg_list = lf_rob_details[:10]["msg"].to_list()
# ram_msg_list =ram_msg["msg"].to_list()

ram_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).collect()

# # reverse_msg_list = lf_rob_details.lazy().sort("mean_num_xact",descending=True).select(pl.col("msg"))[-10:].to_list()
# # reverse_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).collect()


In [13]:
ram_msg_list

['00003',
 '00013',
 '00023',
 '00033',
 '00103',
 '10003',
 '10013',
 '00022',
 '00113',
 '10023']

OPTの各最適だったmsg回数のリスト

In [14]:
opt=lf_opt.sort("max_mean_num_xact",descending=True).collect()

In [15]:
lf_opt_list=(lf_opt
             .sort("max_mean_num_xact",descending=True)).group_by("argmax_msg").len().sort("len",descending=True).collect()
lf_opt_list


argmax_msg,len
str,u32
"""00000""",77
"""00003""",67
"""00013""",36
"""10003""",27
"""00033""",21
…,…
"""02003""",1
"""31103""",1
"""30002""",1


その各msgの全体の結果平均

In [16]:
opt_len_mean=(lf_opt.lazy().join(lf_opt_list.lazy(),left_on=pl.col("argmax_msg"),right_on=pl.col("argmax_msg"),how="left")
              .group_by("argmax_msg").agg(pl.col("max_mean_num_xact").mean(),pl.col("len").first())
              .sort("len",descending=True)
              )
opt_len_mean.collect()[:10]

argmax_msg,max_mean_num_xact,len
str,f64,u32
"""00000""",0.0,77
"""00003""",135.903433,67
"""00013""",300.663056,36
"""10003""",81.268148,27
"""00033""",349.211905,21
"""00023""",224.676471,17
"""10000""",0.0,16
"""10013""",100.52,13
"""20003""",129.2825,8


全体の個人行動回数・比率

In [19]:
ratio_xact_share_per = (user_analysis.lazy()
.select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")).collect())
xact_per_mean = (user_analysis.lazy().select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users")).collect())
share_per_mean = (user_analysis.lazy().select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users")).collect())

In [20]:
concat = pl.concat([xact_per_mean,share_per_mean,ratio_xact_share_per],how="horizontal")
concat

/var/folders/bk/xcfgqxl50_l158g0lbjt3_4r0000gn/T/ipykernel_93552/68081561.py:1: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  concat = pl.concat([xact_per_mean,share_per_mean,ratio_xact_share_per],how="horizontal")


mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.10943,0.151174,0.723865


ram適用前と適用後の個人行動回数・比率

In [21]:
ram_ratio = (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
        .select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")))
ram_mean_xact_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"))
ram_mean_share_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"))
ram = (
    pl.concat([ram_mean_xact_of_users, ram_mean_share_of_users, ram_ratio], how="horizontal").collect())
ram


/var/folders/bk/xcfgqxl50_l158g0lbjt3_4r0000gn/T/ipykernel_93552/1054021564.py:6: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  pl.concat([ram_mean_xact_of_users, ram_mean_share_of_users, ram_ratio], how="horizontal").collect())


mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.152442,0.162715,0.936866


全体の行動回数・比率

In [ ]:
xact_share=(user_analysis.lazy()
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending=True).collect(engine="streaming")
)
xact_share_mean= xact_share.select(pl.col("xact").mean(),pl.col("share").mean())
xact_share_ratio= xact_share.select((pl.col("xact").mean()/pl.col("share").mean()).alias("xact_share"))
# combine1 = pl.concat([xact_share_mean,xact_share_ratio],how="horizontal")


In [23]:
combine1 = pl.concat([xact_share_mean,xact_share_ratio],how="horizontal")
combine1

/var/folders/bk/xcfgqxl50_l158g0lbjt3_4r0000gn/T/ipykernel_93552/1892328318.py:1: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  combine1 = pl.concat([xact_share_mean,xact_share_ratio],how="horizontal")


xact,share,xact_share
f64,f64,f64
40.865631,56.454788,0.723865


In [24]:
ram_xact_share= (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending = True)
).collect(engine="streaming")
ram_xact_share

msg,xact,share
str,f64,f64
"""00003""",95.558173,126.844615
"""00013""",94.809351,103.805721
"""00023""",92.521947,83.708245
"""00033""",89.172332,67.710625
"""00103""",86.151538,103.092428
"""10003""",85.56851,109.046971
"""10013""",84.385168,87.774976
"""00022""",83.956394,102.555
"""00113""",83.603413,82.173413


ram適用後の全体行動回数・比率

In [25]:
ram_xact_share_mean= ram_xact_share.select(pl.col("xact").mean(),pl.col("share").mean())
ram_xact_share_ratio=ram_xact_share.select([(pl.col("xact").mean()/pl.col("share").mean()).alias("xact/share")])
combine = pl.concat([ram_xact_share_mean,ram_xact_share_ratio],how="horizontal")
combine

/var/folders/bk/xcfgqxl50_l158g0lbjt3_4r0000gn/T/ipykernel_93552/2046096511.py:3: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  combine = pl.concat([ram_xact_share_mean,ram_xact_share_ratio],how="horizontal")


xact,share,xact/share
f64,f64,f64
87.826339,93.744798,0.936866
